<a href="https://colab.research.google.com/github/cocyten27/Modulo_5/blob/main/M5_2_Autoencoder_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torchinfo

In [ ]:
import pip
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
from torchinfo import summary
torch.manual_seed(0)

In [ ]:
def compression_factor(depth, in_channels, latent_channels, spatial_dims):
    # each axis shrinks by axis_reduction (e.g. depth=2 -> axis_reduction=4)
    axis_reduction = 2 ** depth
    # spatial_dims axes shrink at the same time, so the element count compounds: axis_reduction**spatial_dims
    spatial_factor = axis_reduction ** spatial_dims
    channel_factor = in_channels / latent_channels
    return axis_reduction, spatial_factor, channel_factor, spatial_factor * channel_factor


def print_compression(name, depth, in_channels, latent_channels, spatial_dims):
    """Print the spatial vs channel breakdown; torchinfo.summary already covers shapes and params."""
    axis, spatial, channel, total = compression_factor(depth, in_channels, latent_channels, spatial_dims)
    print(f"{name} compression factor (depth={depth}): axis /{axis:.0f} -> spatial x{spatial:.2f} (elements) * channels x{channel:.2f} = x{total:.2f}")


def show_architecture(ae, x):
    """Print the full layer-by-layer architecture, input/output shapes and parameter counts."""
    _ = summary(ae, input_data=x, col_names=("input_size", "output_size", "num_params"), verbose=1, depth=2)


In [ ]:
class DownBlock1D(nn.Module):     """MLP en... de Germán Alonso Pinedo Díaz"""

class DownBlock1D(nn.Module):
    """MLP encoder step: merge pairs of samples, halving the length."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(2 * in_channels, out_channels),
            nn.ReLU(inplace=True),
            nn.LayerNorm(out_channels),
        )

    def forward(self, x):
        # x: [batch, channels, length]
        batch, channels, length = x.shape
        if length % 2 != 0:
            raise ValueError("DownBlock1D expects an even length.")
        x = x.transpose(1, 2)                    # [batch, length, channels]
        x = x.reshape(batch, length // 2, 2 * channels)
        x = self.block(x)                        # [batch, length // 2, out_channels]
        return x.transpose(1, 2)                 # [batch, out_channels, length // 2]


class UpBlock1D(nn.Module):
    """MLP decoder step: split each sample into two, doubling the length."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.out_channels = out_channels
        self.proj = nn.Linear(in_channels, 2 * out_channels)
        self.act = nn.ReLU(inplace=True)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        # x: [batch, channels, length]
        batch, _, length = x.shape
        x = x.transpose(1, 2)                    # [batch, length, channels]
        x = self.act(self.proj(x))               # [batch, length, 2 * out_channels]
        x = x.reshape(batch, length * 2, self.out_channels)
        x = self.norm(x)
        return x.transpose(1, 2)                 # [batch, out_channels, length * 2]

In [ ]:
class Autoencoder1D(nn.Module):
    """Stacks `depth` DownBlock/UpBlock pairs."""
    def __init__(self, in_channels, latent_channels, depth=1):
        super().__init__()
        self.depth = depth
        channels = [in_channels] + [latent_channels] * depth
        self.encoder = nn.ModuleList(
            [DownBlock1D(channels[i], channels[i + 1]) for i in range(depth)]
        )

        # depth 2
        self.decoder = nn.ModuleList(
            [UpBlock1D(channels[i + 1], channels[i]) for i in reversed(range(depth))]
        )

    def encode(self, x):
        z = x
        for down in self.encoder:
            z = down(z)
        return z

    def decode(self, z):
        y = z
        for up in self.decoder:
            y = up(y)
        return y

    def forward(self, x):
        return self.decode(self.encode(x))


x1 = torch.randn(4, 1, 1024)

ae1 = Autoencoder1D(in_channels=1, latent_channels=2, depth=4)
show_architecture(ae1, x1)
print_compression("1D", depth=4, in_channels=3, latent_channels=2, spatial_dims=1)

#ae1_deep = Autoencoder1D(in_channels=3, latent_channels=2, depth=2)
#show_architecture(ae1_deep, x1)
#print_compression("1D (depth=2)", depth=2, in_channels=3, latent_channels=2, spatial_dims=1)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
Autoencoder1D                            [4, 1, 1024]              [4, 1, 1024]              --
├─ModuleList: 1-1                        --                        --                        --
│    └─DownBlock1D: 2-1                  [4, 1, 1024]              [4, 2, 512]               10
│    └─DownBlock1D: 2-2                  [4, 2, 512]               [4, 2, 256]               14
│    └─DownBlock1D: 2-3                  [4, 2, 256]               [4, 2, 128]               14
│    └─DownBlock1D: 2-4                  [4, 2, 128]               [4, 2, 64]                14
├─ModuleList: 1-2                        --                        --                        --
│    └─UpBlock1D: 2-5                    [4, 2, 64]                [4, 2, 128]               16
│    └─UpBlock1D: 2-6                    [4, 2, 128]               [4, 2, 256]               16
│    └─UpBlock1D: 2-7              

Autoencoder 2D

In [ ]:
class ConvBlock2D(nn.Module):
    """Conv -> Activation -> Norm, spatial size unchanged (stride=1)."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        return self.block(x)


class DownBlock2D(nn.Module):
    """Encoder step: ConvBlock followed by pooling, which halves height and width."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = ConvBlock2D(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        return self.pool(self.conv(x))


class UpBlock2D(nn.Module):
    """Decoder step: upsampling (linear/bilinear or ConvTranspose) followed by ConvBlock."""
    def __init__(self, in_channels, out_channels, mode="convtranspose"):
        super().__init__()
        if mode == "convtranspose":
            self.upsample = nn.ConvTranspose2d(in_channels, in_channels, kernel_size=2, stride=2)
        elif mode == "linear":
            self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        else:
            raise ValueError(f"Unknown upsampling mode: {mode}")
        self.conv = ConvBlock2D(in_channels, out_channels)

    def forward(self, x):
        return self.conv(self.upsample(x))

In [ ]:
class Autoencoder2D(nn.Module):
    """Stacks depth DownBlock/UpBlock pairs."""
    def __init__(self, in_channels, latent_channels, depth=1, up_mode="convtranspose"):
        super().__init__()
        self.depth = depth
        self.up_mode = up_mode

        # Similar channel progression as Autoencoder1D
        # First encoder block: in_channels -> latent_channels
        # Subsequent encoder blocks: latent_channels -> latent_channels
        # Last decoder block: latent_channels -> in_channels
        # Other decoder blocks: latent_channels -> latent_channels
        channels = [in_channels] + [latent_channels] * depth

        self.encoder = nn.ModuleList(
            [DownBlock2D(channels[i], channels[i+1]) for i in range(depth)]
        )

        self.decoder = nn.ModuleList(
            [UpBlock2D(channels[i+1], channels[i], mode=self.up_mode) for i in reversed(range(depth))]
        )

    def encode(self, x):
        z = x
        for down in self.encoder:
            z = down(z)
        return z

    def decode(self, z):
        y = z
        for up in self.decoder:
            y = up(y)
        return y

    def forward(self, x):
        return self.decode(self.encode(x))


x2 = torch.randn(2, 1, 28, 28)

ae2 = Autoencoder2D(in_channels=1, latent_channels=4, depth=1)
show_architecture(ae2, x2)
print_compression("2D", depth=1, in_channels=1, latent_channels=4, spatial_dims=2)

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #
Autoencoder2D                                 [2, 1, 28, 28]            [2, 1, 28, 28]            --
├─ModuleList: 1-1                             --                        --                        --
│    └─DownBlock2D: 2-1                       [2, 1, 28, 28]            [2, 4, 14, 14]            48
├─ModuleList: 1-2                             --                        --                        --
│    └─UpBlock2D: 2-2                         [2, 4, 14, 14]            [2, 1, 28, 28]            107
Total params: 155
Trainable params: 155
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.23
Input size (MB): 0.01
Forward/backward pass size (MB): 0.18
Params size (MB): 0.00
Estimated Total Size (MB): 0.18
2D compression factor (depth=1): axis /2 -> spatial x4.00 (elements) * channels x0.25 = x1.00


## Autoencoder 3D

In [ ]:
class ConvBlock3D(nn.Module):
    """Conv -> Activation -> Norm, spatial size unchanged (stride=1)."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm3d(out_channels),
        )

    def forward(self, x):
        return self.block(x)


class DownBlock3D(nn.Module):
    """Encoder step: ConvBlock followed by pooling, which halves depth, height, and width."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = ConvBlock3D(in_channels, out_channels)
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)

    def forward(self, x):
        return self.pool(self.conv(x))


class UpBlock3D(nn.Module):
    """Decoder step: upsampling (linear/bilinear/trilinear or ConvTranspose) followed by ConvBlock."""
    def __init__(self, in_channels, out_channels, mode="convtranspose"):
        super().__init__()
        if mode == "convtranspose":
            self.upsample = nn.ConvTranspose3d(in_channels, in_channels, kernel_size=2, stride=2)
        elif mode == "linear":
            self.upsample = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False)
        else:
            raise ValueError(f"Unknown upsampling mode: {mode}")
        self.conv = ConvBlock3D(in_channels, out_channels)

    def forward(self, x):
        return self.conv(self.upsample(x))

In [ ]:
class Autoencoder3D(nn.Module):
    """Stacks depth DownBlock/UpBlock pairs."""
    def __init__(self, in_channels, latent_channels, depth=1, up_mode="convtranspose"):
        super().__init__()
        self.depth = depth
        self.up_mode = up_mode

        channels = [in_channels] + [latent_channels] * depth

        self.encoder = nn.ModuleList(
            [DownBlock3D(channels[i], channels[i+1]) for i in range(depth)]
        )

        self.decoder = nn.ModuleList(
            [UpBlock3D(channels[i+1], channels[i], mode=self.up_mode) for i in reversed(range(depth))]
        )

    def encode(self, x):
        z = x
        for down in self.encoder:
            z = down(z)
        return z

    def decode(self, z):
        y = z
        for up in self.decoder:
            y = up(y)
        return y

    def forward(self, x):
        return self.decode(self.encode(x))


x3 = torch.randn(2, 1, 16, 16, 16) # Example 3D input: batch, channels, depth, height, width

ae3 = Autoencoder3D(in_channels=1, latent_channels=4, depth=1)
show_architecture(ae3, x3)
print_compression("3D", depth=1, in_channels=1, latent_channels=4, spatial_dims=3)

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #
Autoencoder3D                                 [2, 1, 16, 16, 16]        [2, 1, 16, 16, 16]        --
├─ModuleList: 1-1                             --                        --                        --
│    └─DownBlock3D: 2-1                       [2, 1, 16, 16, 16]        [2, 4, 8, 8, 8]           120
├─ModuleList: 1-2                             --                        --                        --
│    └─UpBlock3D: 2-2                         [2, 4, 8, 8, 8]           [2, 1, 16, 16, 16]        243
Total params: 363
Trainable params: 363
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 2.89
Input size (MB): 0.03
Forward/backward pass size (MB): 0.92
Params size (MB): 0.00
Estimated Total Size (MB): 0.95
3D compression factor (depth=1): axis /2 -> spatial x8.00 (elements) * channels x0.25 = x2.00
